# Streamlit Runner untuk Google Colab
Notebook ini dipakai untuk menjalankan dashboard Streamlit dari Colab setelah file `streamlit_app.py` dan `requirements.txt` tersedia di repository GitHub.

## 1. Clone repository dan install dependency
Jalankan cell berikut. Kalau kamu sudah upload file secara manual ke Colab, bagian `git clone` bisa dilewati.

In [ ]:
!git clone https://github.com/depmkeu/big-data-sentiment-analysis.git
%cd big-data-sentiment-analysis
!pip install -r requirements.txt

## 2. Jalankan Streamlit menggunakan localtunnel
URL dashboard akan muncul pada output cell. Karena Colab tidak bisa expose port secara langsung, localtunnel dipakai untuk membuat link sementara.

In [ ]:
!npm install -g localtunnel
!streamlit run streamlit_app.py --server.port 8501 & npx localtunnel --port 8501

## 3. Export hasil prediksi dari notebook utama
Jalankan cell ini di notebook utama setelah variabel `predictions` terbentuk. File CSV hasil ekspor bisa digunakan untuk dashboard.

In [ ]:
from pyspark.sql.functions import when, col

predictions_export = predictions.withColumn(
    "predicted_sentiment",
    when(col("prediction") == 0.0, "positive")
    .when(col("prediction") == 1.0, "negative")
    .when(col("prediction") == 2.0, "neutral")
    .otherwise("unknown")
)

predictions_export.select(
    "id", "keyword", "subdomain", "final label", "prediction", "predicted_sentiment"
).coalesce(1).write.mode("overwrite").option("header", True).csv("/content/sentiment_results_export")